# Decision Tree or Random Forest? An experiment with evidence
**CS 4120 · Machine Learning and Data Mining · October 21, 2026**  
University of Prince Edward Island

**Name(s):** _Write here_  
**Lecture companion:** Random Forests and Experimentation in Machine Learning

Today you will compare a Decision Tree (DT) with a Random Forest (RF), tune both with grid search, and justify a model choice using cross-validation. The code provides the experiment and plotting tools. **You provide the predictions, decisions and reflections.**

### By the end, you should be able to
- Explain why different trees can help a forest generalize.
- Compare models using the same data splits, metric and preprocessing pipeline.
- Read validation plots without mistaking training performance for generalization.
- Separate model selection from final evaluation.
- Explain what you learned, what changed your mind, and what remains uncertain.

### How to follow along
Upload this `.ipynb` file at [Google Colab](https://colab.research.google.com/) using **File → Upload notebook**, then save your own copy. Use a standard CPU runtime. No GPU, Drive mount, CSV upload, or paid account is required. Run cells in order with **Shift + Enter**. Double-click a text cell to write your answers.

Suggested class plan: 10 minutes for setup and predictions, 15 for the baseline and pipeline, 20 for tuning and plots, 15 for the model decision and evaluation, and 15 for reflection. The extension is optional.

**Important:** the final test cell stays closed until you record a choice and explanation. Running all cells with the supplied defaults does not open the test set.

## 1. Predict before running
*Lecture connection: slides 2–17.*

1. Which model do you expect to perform better on unseen examples, and why?
2. Which model do you expect to be easier to explain to another person?
3. If a model gets perfect training performance, what additional evidence would you need?

**My prediction:** _Write 2–4 sentences before running the experiment._

## 2. Setup
Colab includes these libraries. We use the installed versions and record them rather than changing the environment during class. The lecture's numerical example used scikit-learn 1.5.2. Small differences across library versions are possible.

`random_state` controls repeatability. It does not guarantee that one split represents the whole population.

In [ ]:
import sys
import json
import platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

from IPython.display import display
from sklearn.base import clone
from sklearn.datasets import load_wine
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV,
    cross_validate, ParameterGrid
)
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree

SEED = 42
N_JOBS = 1  # Predictable CPU use. The search may take a minute or two.
SCORING = "f1_macro"
COLORS = {"Baseline": "#777B98", "Decision Tree": "#BF5105",
          "Random Forest": "#5054C2"}
plt.rcParams.update({"figure.dpi": 110, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
versions = {"python": platform.python_version(), "numpy": np.__version__,
            "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
            "scikit-learn": sklearn.__version__}
display(pd.Series(versions, name="Version"))

## 3. The right dataset for this lecture
We use scikit-learn's **Wine classification dataset**, the same dataset as the lecture experiment: 178 samples, 13 numeric chemical measurements, and 3 cultivar classes. This is **not** the Wine Quality dataset and we are **not** predicting a quality score.

The target values 0, 1 and 2 identify classes. They are not ordered ratings. For this classroom exercise, we treat rows as independent observations. A real collection with repeated vineyards, batches or time periods would require a split that respects that structure.

[Dataset documentation and original-data attribution](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html)

We reserve the test set immediately. All exploration below uses **training data only**.

In [ ]:
wine = load_wine(as_frame=True)
X = wine.data
# Labels stay separate from the input features.
y = wine.target
CLASS_LABELS = np.unique(y)
CLASS_NAMES = [str(name) for name in wine.target_names]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
assert set(X_train.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_test) == len(X)
print(f"Dataset: {X.shape[0]} rows, {X.shape[1]} numeric features")
print(f"Development/training rows: {len(X_train)}")
print(f"Reserved test rows: {len(X_test)} (do not inspect yet)")
display(X_train.head())
display(X_train.describe().round(2))
print("Missing training values:", int(X_train.isna().sum().sum()))

### Plot helper 1: class balance
The helper functions are reusable. Read their short descriptions, then focus on interpreting the plots. You do not need to rewrite the plotting code.

In [ ]:
def plot_class_balance(labels, class_names):
    """Count training examples in each class; never pass the test labels here."""
    counts = pd.Series(labels).value_counts().reindex(
        range(len(class_names)), fill_value=0
    )
    fig, ax = plt.subplots(figsize=(7, 3.5))
    bars = ax.bar(class_names, counts, color=["#171952", "#5054C2", "#BF5105"])
    ax.bar_label(bars, padding=3)
    ax.set(ylabel="Training examples", title="Class balance in the training data")
    ax.set_ylim(0, counts.max() * 1.2)
    fig.tight_layout()
    plt.show()

plot_class_balance(y_train, CLASS_NAMES)

**Pause and interpret**

- Which class has the most training examples? Does that make it more important?
- Why should we compare our models with a simple baseline?
- Why might a stratified random split be inappropriate for repeated measurements from one vineyard?

**My answers:** _Write here._

## 4. A metric and a fair comparison
*Lecture connection: slides 18–24.*

Our primary metric is **macro F1**, where larger values are better. We calculate an F1 score for each class and average the three scores equally.

- **Precision:** among predictions of one class, how many are correct?
- **Recall:** among actual examples of that class, how many did we find?
- **F1:** the harmonic mean of precision and recall, `2 × P × R / (P + R)`.

Equal weighting gives each cultivar the same contribution to macro F1, regardless of class frequency. We also report accuracy at the end, but we do not change our selection metric after seeing which model wins.

**Protocol:** same development rows, same five folds, same metric, and the same preprocessing. We compare the best searched configuration from each model family. RF has a larger search budget here; this is a comparison of these two specified searches, not proof that one algorithm is universally better.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# Materialize the splits once, so every model uses exactly the same row positions.
folds = list(cv.split(X_train, y_train))
assert all(set(train_idx).isdisjoint(valid_idx) for train_idx, valid_idx in folds)
assert sorted(np.concatenate([v for _, v in folds]).tolist()) == list(range(len(X_train)))

display(pd.DataFrame([
    {"Fold": i + 1, "Fit rows": len(tr), "Validation rows": len(va)}
    for i, (tr, va) in enumerate(folds)
]))

## 5. Baseline: always predict the most frequent class
A baseline asks whether our more complex models add useful information. Its predictions can have nontrivial accuracy even though it ignores all chemical measurements.

In [ ]:
baseline_result = cross_validate(
    DummyClassifier(strategy="most_frequent"),
    X_train, y_train, cv=folds, scoring=SCORING,
    return_train_score=True, n_jobs=N_JOBS
)
print(f"Baseline mean CV macro F1: {baseline_result['test_score'].mean():.3f}")
print("Fold scores:", np.round(baseline_result["test_score"], 3))

## 6. Pipelines keep learning inside the fold
Each pipeline contains a median imputer and a classifier. During each CV run, the imputer learns its medians from that fold's training portion. It uses those medians to transform the validation portion.

Wine has no missing values, so the imputer currently leaves the values unchanged. Keeping it inside the pipeline demonstrates the right workflow when missing numeric measurements appear. Trees generally do not need feature scaling. Categorical strings in a different dataset would need suitable encoding.

**Explain before proceeding:** What would leak if we fitted an imputer or selected features using all rows before splitting?

**My explanation:** _Write here._

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "Random Forest": RandomForestClassifier(random_state=SEED, n_jobs=1)
}

def make_pipeline(model):
    """A fresh preprocessing/model pair for each fit."""
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("model", clone(model))
    ])

make_pipeline(models["Decision Tree"])

## 7. What will the grid search vary?
*Lecture connection: slides 11–12 and 24–28.*

| Setting | Meaning | Model |
|---|---|---|
| `max_depth` | Maximum depth of each tree; `None` removes that explicit limit | DT and RF |
| `min_samples_leaf` | Minimum training examples allowed in a leaf | DT and RF |
| `n_estimators` | Number of trees | RF |
| `max_features` | Candidate features at each split; `"sqrt"` uses 3 of 13 here, `1.0` allows all | RF |

`model__` refers to the pipeline step called `model`. The training process learns the actual thresholds and leaf predictions.

**Predict:** Which setting might reduce a gap between training and validation performance? Why?

**My answer:** _Write here._

In [ ]:
grids = {
    "Decision Tree": {
        "model__max_depth": [3, 6, None],
        "model__min_samples_leaf": [1, 4]
    },
    "Random Forest": {
        "model__n_estimators": [100, 300],
        "model__max_depth": [3, 6, None],
        "model__min_samples_leaf": [1, 4],
        "model__max_features": ["sqrt", 1.0]
    }
}
counts = {name: len(ParameterGrid(grid)) for name, grid in grids.items()}
print("Configurations:", counts)
print("CV fits:", sum(counts.values()) * len(folds))
print("Plus one refit for each model family after its search.")

### Run the two searches
This uses 150 CV fits plus two refits. Each search refits its own winning pipeline on all 142 development rows. Both refitted pipelines are available for inspection; neither has used the reserved test rows.

The search calls its validation scores `mean_test_score`. **Inside `cv_results_`, “test” means the CV validation fold, not our held-out `X_test`.**

In [ ]:
if globals().get("_test_opened", False):
    raise RuntimeError("The test set is already open. Do not tune again using its feedback.")

searches = {}
for name, model in models.items():
    search = GridSearchCV(
        make_pipeline(model), grids[name], scoring=SCORING,
        cv=folds, n_jobs=N_JOBS, refit=True,
        return_train_score=True, error_score="raise"
    )
    search.fit(X_train, y_train)
    searches[name] = search
    print(name, "best mean CV F1:", round(search.best_score_, 3))
    print("Chosen parameters:", search.best_params_)

## 8. A comparison table for decisions
The helper extracts the winning candidate within each family and its scores on the shared folds. The fold standard deviation describes variation across folds. It is **not** a confidence interval or a significance test.

The same folds helped us select the best settings, so these are **selection results**, not independent estimates of the tuned procedure's performance.

In [ ]:
def summarize_searches(searches, baseline, n_folds):
    """Return a summary and matched validation-fold scores for each winner."""
    rows = [{
        "Model": "Baseline", "CV mean": np.mean(baseline["test_score"]),
        "CV SD": np.std(baseline["test_score"], ddof=0),
        "Train mean": np.mean(baseline["train_score"]),
        "Mean fit seconds": np.mean(baseline["fit_time"])
    }]
    fold_scores = {"Baseline": np.asarray(baseline["test_score"])}
    for name, search in searches.items():
        result, best = search.cv_results_, search.best_index_
        scores = np.array([result[f"split{k}_test_score"][best]
                           for k in range(n_folds)])
        fold_scores[name] = scores
        rows.append({
            "Model": name, "CV mean": result["mean_test_score"][best],
            "CV SD": result["std_test_score"][best],
            "Train mean": result["mean_train_score"][best],
            "Mean fit seconds": result["mean_fit_time"][best]
        })
    summary = pd.DataFrame(rows).set_index("Model")
    summary["Train minus CV"] = summary["Train mean"] - summary["CV mean"]
    return summary, pd.DataFrame(fold_scores, index=range(1, n_folds + 1))

summary, fold_scores = summarize_searches(searches, baseline_result, len(folds))
fold_scores.index.name = "Fold"
display(summary.round(4))
display(fold_scores.round(4))
print("Fit times depend on the machine and are rough costs, not universal benchmarks.")

### Plot helpers 2 and 3: mean performance and matched folds
The first plot shows average validation performance and fold SD. The second connects the scores across shared folds, making it easier to see whether one difficult fold affects both models.

In [ ]:
def plot_cv_comparison(summary, fold_scores):
    """Show CV mean ± fold SD and the matched per-fold observations."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    names = list(summary.index)
    for i, name in enumerate(names):
        axes[0].errorbar(i, summary.loc[name, "CV mean"],
                         yerr=summary.loc[name, "CV SD"], fmt="o",
                         color=COLORS[name], capsize=6, markersize=9)
    axes[0].set_xticks(range(len(names)), names, rotation=12)
    axes[0].set(title="Mean CV macro F1 and fold SD", ylabel="Macro F1", ylim=(0, 1.05))
    for name in fold_scores:
        axes[1].plot(fold_scores.index, fold_scores[name], "o-",
                     label=name, color=COLORS[name])
    axes[1].set(title="Same validation folds", xlabel="Fold", ylabel="Macro F1",
                xticks=fold_scores.index, ylim=(0, 1.05))
    axes[1].legend(loc="lower right")
    for ax in axes:
        ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    plt.show()

plot_cv_comparison(summary, fold_scores)
paired_difference = fold_scores["Random Forest"] - fold_scores["Decision Tree"]
display(paired_difference.rename("RF minus DT macro F1").to_frame().round(4))

**Evidence checkpoint**

1. Record each model's mean CV macro F1 and fold SD.
2. Does the same model win on every fold? Cite the fold differences.
3. Which model has more variable validation performance?
4. Why is a small difference between means insufficient on its own?

**My evidence:** _Use at least two numbers and one observation from the plots._

### Plot helper 4: training and validation together
A large training–validation gap can suggest overfitting. Low performance on both can suggest underfitting, but neither diagnosis follows from the gap alone. Consider the task, features, sample size and metric.

In [ ]:
def plot_train_validation(summary):
    """Compare fold-training scores with fold-validation scores."""
    names = [name for name in summary.index if name != "Baseline"]
    x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - 0.18, summary.loc[names, "Train mean"], 0.36,
           label="Training mean", color="#A8ACDC")
    ax.bar(x + 0.18, summary.loc[names, "CV mean"], 0.36,
           label="Validation mean", color="#5054C2")
    ax.set(xticks=x, xticklabels=names, ylim=(0, 1.08), ylabel="Macro F1",
           title="Training fit versus validation performance")
    ax.legend(loc="lower right")
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    plt.show()

plot_train_validation(summary)

**My diagnosis:** Which model shows the larger gap? Does the evidence support overfitting, underfitting, or neither clearly? Explain why perfect training performance alone cannot decide this.

## 9. How did the hyperparameters affect the score?
### Plot helper 5: a grid-search heatmap
For DT, the plot shows the full depth × leaf-size grid. For RF, we must fix `n_estimators` and `max_features` to show a two-dimensional slice. The RF plot below fixes those settings at its winning values. **It is a slice, not the whole search.**

In [ ]:
def plot_grid_slice(search, title, fixed=None):
    """Plot depth × leaf size after explicitly fixing other grid dimensions."""
    fixed = fixed or {}
    records = []
    for params, score in zip(search.cv_results_["params"],
                             search.cv_results_["mean_test_score"]):
        if all(params.get(key) == value for key, value in fixed.items()):
            depth = params["model__max_depth"]
            records.append({"Depth": "None" if depth is None else str(depth),
                            "Min leaf": params["model__min_samples_leaf"],
                            "CV F1": score})
    if not records:
        raise ValueError("No candidates match this slice.")
    frame = pd.DataFrame(records)
    if frame.duplicated(["Depth", "Min leaf"]).any():
        raise ValueError("Fix all other varying hyperparameters before plotting.")
    matrix = frame.pivot(index="Min leaf", columns="Depth", values="CV F1")
    depth_order = sorted(matrix.columns, key=lambda v: float("inf") if v == "None" else int(v))
    matrix = matrix.reindex(columns=depth_order).sort_index()
    fig, ax = plt.subplots(figsize=(7, 3.8))
    image = ax.imshow(matrix, cmap="Purples", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(matrix.columns)), matrix.columns)
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    ax.set(xlabel="max_depth", ylabel="min_samples_leaf", title=title)
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            value = matrix.iloc[row, col]
            ax.text(col, row, f"{value:.3f}", ha="center", va="center",
                    color="white" if value > 0.65 else "black")
    fig.colorbar(image, ax=ax, label="Mean CV macro F1")
    fig.tight_layout()
    plt.show()

plot_grid_slice(searches["Decision Tree"], "Decision Tree: full search grid")
rf_params = searches["Random Forest"].best_params_
rf_fixed = {key: rf_params[key] for key in
            ["model__n_estimators", "model__max_features"]}
print("RF slice fixes:", rf_fixed)
plot_grid_slice(searches["Random Forest"], "Random Forest: selected grid slice", rf_fixed)

**Your decision from the heatmaps**

- Did increasing depth always improve validation performance?
- Did allowing smaller leaves always help?
- What does the RF slice leave out?
- If several settings score almost the same, what might favor a simpler or cheaper option?

**My answers:** _Write here. Distinguish an observed result from a general claim._

## 10. Optional inspection: what does the tree look like?
### Plot helper 6: a readable tree preview
This displays the winning DT trained on all development data. To keep labels readable, the figure shows only the top few levels. A truncated drawing does **not** change or prune the fitted model.

In [ ]:
def plot_tree_preview(fitted_pipeline, feature_names, class_names, display_depth=2):
    """Draw a truncated view of a fitted Decision Tree pipeline."""
    tree = fitted_pipeline.named_steps["model"]
    if not isinstance(tree, DecisionTreeClassifier):
        raise TypeError("This helper expects a Decision Tree pipeline.")
    print(f"Actual tree: depth {tree.get_depth()}, leaves {tree.get_n_leaves()}")
    fig, ax = plt.subplots(figsize=(16, 6))
    plot_tree(tree, feature_names=list(feature_names), class_names=list(class_names),
              max_depth=display_depth, filled=True, rounded=True, fontsize=9, ax=ax)
    ax.set_title(f"Decision Tree preview: top {display_depth + 1} levels")
    fig.tight_layout()
    plt.show()

plot_tree_preview(searches["Decision Tree"].best_estimator_, X_train.columns, CLASS_NAMES)

**Read one path:** Choose a visible split and describe its rule in plain language. Why would inspecting one tree from a forest give only a partial explanation of the forest's prediction?

**My answer:** _Write here._

## 11. Optional extension: one justified change, before the test
Skip this section if class time is short. To try it, set `RUN_EXTENSION = True` and write your hypothesis. The supplied example compares **larger leaf sizes** for the Decision Tree. You may choose another small grid.

This extension does not silently replace the main comparison. It displays a separate result for discussion. For today's final evaluation, choose one of the two original searches. If you redesign the main protocol, do it before opening the test set and document the new search budget.

**My hypothesis:** _Changing ___ should affect ___ because ___. The plot or score that would challenge my prediction is ___._

In [ ]:
RUN_EXTENSION = False
if RUN_EXTENSION:
    if globals().get("_test_opened", False):
        raise RuntimeError("Do not tune after inspecting the held-out test result.")
    extension = GridSearchCV(
        make_pipeline(models["Decision Tree"]),
        {"model__max_depth": [3, 6, None], "model__min_samples_leaf": [4, 8, 12]},
        scoring=SCORING, cv=folds, n_jobs=N_JOBS,
        return_train_score=True, error_score="raise"
    ).fit(X_train, y_train)
    print("Original DT mean CV:", searches["Decision Tree"].best_score_)
    print("Extension mean CV:", extension.best_score_)
    print("Extension settings:", extension.best_params_)
    plot_grid_slice(extension, "Extension: larger minimum leaves")
else:
    print("Optional extension skipped.")

## 12. Model decision: stop before the final test
*Lecture connection: slides 29–32.*

Choose **Decision Tree** or **Random Forest** using only the evidence so far. The automatic suggestion below ranks mean CV macro F1. Your explanation must also discuss variation, complexity and limitations. A different choice requires a clear, task-relevant reason.

### Decision record
- **Chosen model:** _Write here._
- **CV evidence:** _Mean, fold SD, and at least one matched-fold observation._
- **Complexity or cost:** _What do you gain or give up?_
- **Why the other model is less suitable here:** _Explain._
- **Uncertainty:** _What can these data and this search not establish?_

After writing your record, set `CHOSEN_MODEL` and `CHOICE_REASON` in the next cell. Then explicitly set `OPEN_FINAL_TEST = True`. Do not adjust settings after seeing the result.

In [ ]:
suggested = summary.loc[list(searches), "CV mean"].idxmax()
print("Highest mean CV macro F1:", suggested)

CHOSEN_MODEL = None  # Replace with "Decision Tree" or "Random Forest".
CHOICE_REASON = ""  # Write your evidence-based justification.
OPEN_FINAL_TEST = False  # Change only after recording your decision.

### Plot helper 7: confusion matrix and final metrics
The confusion matrix shows actual classes in rows and predictions in columns. Diagonal counts are correct predictions. Off-diagonal counts identify errors.

Only the **chosen** pipeline receives a final test evaluation. The RF–DT comparison already happened through shared CV. Testing both and then changing the choice would reuse the final test as selection data.

In [ ]:
def show_final_evaluation(y_true, prediction, labels, class_names, model_name):
    """Display a cached final prediction; this function does not fit a model."""
    metrics = pd.Series({
        "Macro F1": f1_score(y_true, prediction, average="macro", zero_division=0),
        "Accuracy": accuracy_score(y_true, prediction),
        "Test rows": len(y_true)
    }, name=model_name)
    display(metrics.to_frame())
    report = classification_report(y_true, prediction, labels=labels,
                                   target_names=class_names, output_dict=True,
                                   zero_division=0)
    display(pd.DataFrame(report).T.drop(index="accuracy", errors="ignore").round(3))
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_true, prediction, labels=labels, display_labels=class_names,
        cmap="Purples", colorbar=False, ax=ax
    )
    ax.set_title(f"Final held-out evaluation: {model_name}")
    fig.tight_layout()
    plt.show()
    return metrics

if not OPEN_FINAL_TEST:
    print("Test set stays closed. Complete your decision record first.")
elif CHOSEN_MODEL not in searches or len(CHOICE_REASON.strip()) < 20:
    print("Choose a valid model and write a meaningful justification before opening the test.")
else:
    requested_choice = {
        "model": CHOSEN_MODEL,
        "settings": searches[CHOSEN_MODEL].best_params_,
        "reason": CHOICE_REASON
    }
    if globals().get("_test_opened", False):
        if requested_choice != frozen_choice:
            raise RuntimeError("The choice is frozen. Do not switch models after seeing the test.")
        print("Showing cached predictions from the existing final evaluation.")
    else:
        frozen_choice = requested_choice.copy()
        selected_pipeline = searches[CHOSEN_MODEL].best_estimator_
        # Already refitted on all development rows by GridSearchCV.
        final_prediction = selected_pipeline.predict(X_test)
        _test_opened = True
    final_metrics = show_final_evaluation(
        y_test, final_prediction, CLASS_LABELS, CLASS_NAMES, frozen_choice["model"]
    )

## 13. Interpret the final result
1. How does your final test macro F1 compare with the chosen model's mean CV score?
2. Which errors appear in the confusion matrix? If there are none, what does that tell you **only about these 36 observations**?
3. Why does a perfect score on a small test set not establish perfect future performance?
4. If the result disappoints you, why should you avoid reopening the search and selecting a new model using this test set?

**My interpretation:** _Write one paragraph supported by your actual result._

**Connection to the lecture:** This repeats a dataset and split whose results appear in the slides. That makes it useful for learning, but its test result is not new, unseen scientific evidence for the class. In a new project, reserve genuinely unused data. A runtime restart does not make previously observed test results unknown again.

## 14. Reflection: what did you learn, and why?
Answer with specific evidence from your run. “RF is better” is not enough.

| Prompt | Your reflection |
|---|---|
| What did I believe before the experiment? | _Write here._ |
| Which result or plot supported or changed that belief? Why? | _Include numbers or identify a plot._ |
| What did I learn about one tree versus many different trees? | _Explain the role of randomness and averaging._ |
| What did I learn about hyperparameters? | _Distinguish them from learned splits._ |
| Why did the pipeline belong inside CV? | _Describe one leakage pathway._ |
| What did comparing shared folds reveal that one score hid? | _Use your own results._ |
| What would I do differently in my next experiment? Why? | _Propose one justified improvement._ |
| What remains uncertain? | _Discuss sample size, split assumptions, search budget, or new populations._ |

### Final claim: 100–150 words
Write a recommendation that includes the chosen model, validation evidence, final evaluation, a tradeoff, and a limitation. Explain **why** the evidence supports your decision without claiming that one model always wins.

**My recommendation:** _Write here._

### Partner check
Exchange notebooks. Ask your partner to identify one claim that is directly supported by evidence and one claim that needs a qualification.

## 15. What to hand in
- Your saved notebook with executed plots and tables.
- Your initial prediction, evidence checkpoints, and model-decision record.
- The final confusion matrix for your chosen model, plus your interpretation.
- Your completed reflection and 100–150 word recommendation.

The supplied notebook can run from top to bottom without opening the test set. After you complete the decision cell, it can also reproduce your recorded final evaluation. Reproducing a result does not create fresh test evidence.

### Reproducibility record
The next cell saves a small JSON report containing the split settings, library versions, searched hyperparameters and CV summary. If you opened the test, it also saves your frozen choice and final metrics. This helps you retain the exact experiment alongside your notebook.

In [ ]:
report = {
    "lecture_date": "2026-10-21", "dataset": "sklearn.datasets.load_wine",
    "seed": SEED, "test_fraction": 0.20, "training_rows": len(X_train),
    "test_rows": len(X_test), "cv": f"5-fold stratified, shuffled, seed {SEED}",
    "metric": SCORING, "versions": versions, "grids": grids,
    "cv_summary": summary.to_dict(orient="index"),
    "best_settings": {name: search.best_params_ for name, search in searches.items()}
}
if globals().get("_test_opened", False):
    report["frozen_choice"] = frozen_choice
    report["final_metrics"] = {key: float(value) for key, value in final_metrics.items()}
with open("rf_dt_experiment_record.json", "w", encoding="utf-8") as file:
    json.dump(report, file, indent=2)
print("Saved rf_dt_experiment_record.json in the notebook's working directory.")
print("In Colab, use the Files sidebar to download it if required.")

## References and further questions
- [Wine dataset and provenance](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html)
- [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- [Pipelines and data leakage](https://scikit-learn.org/stable/common_pitfalls.html)
- [Cross-validation strategies](https://scikit-learn.org/stable/modules/cross_validation.html)
- [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

For a later project: nested CV separates inner hyperparameter selection from outer evaluation of the entire tuning procedure. Feature importance can help inspect a fitted model, but importance does not establish causation. Neither topic is required to complete today's exercise.